# Data merge and feature engineering


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
from tqdm import tqdm
import pymorphy3
from collections import Counter

import os
from pathlib import Path
import json

In [2]:
df_weather = pd.read_csv("../data/weather_processed.csv")
df_war_events = pd.read_csv("../data/war_events_processed.csv")
df_isw = pd.read_csv("../data/isw_processed.csv")
df_isw_matrix = pd.read_csv("../data/isw_processed_svd.csv")
df_tg = pd.read_csv("../data/telegram_processed.csv")

In [3]:
df_weather['datetime_hour'] = pd.to_datetime(df_weather['datetime_hour'], errors="coerce")
df_war_events['datetime_hour'] = pd.to_datetime(df_war_events['datetime_hour'], errors="coerce")

In [4]:
df_final = pd.merge(
    df_weather, 
    df_war_events[['datetime_hour', 'region_id', 'alarm_active', 'alarm_minutes_in_hour']], 
    on=['datetime_hour', 'region_id'], 
    how='left'
)
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
175917,2022-12-26 10:00:00,24,48.2924,25.9355,1.1,0.5,95.8,0.316,100.0,8.33,...,10,Chernivtsi,Europe/Kiev,2022-12-26,08:06:35,16:27:06,10:00:00,Чернівецька,0,0.000000
594345,2024-12-21 23:00:00,11,48.5085,32.2656,4.3,3.6,95.8,1.000,100.0,4.17,...,23,Kropyvnytskyi,Europe/Kiev,2024-12-21,07:40:26,15:58:07,23:00:00,Кіровоградська,1,60.000000
253864,2023-05-10 19:00:00,19,49.5527,25.5889,9.2,-2.0,49.4,0.000,0.0,0.00,...,19,Ternopil,Europe/Kiev,2023-05-10,05:41:18,20:47:51,19:00:00,Тернопільська,0,0.000000
6017,2022-03-06 10:00:00,20,50.0042,36.2358,-2.7,-8.5,65.1,0.000,0.0,0.00,...,10,Kharkiv,Europe/Kiev,2022-03-06,06:08:24,17:25:11,10:00:00,Харківська,0,0.000000
164044,2022-12-05 20:00:00,6,50.2536,28.6654,-7.0,-9.4,83.6,0.000,0.0,0.00,...,20,Zhytomyr,Europe/Kiev,2022-12-05,07:48:08,16:03:29,20:00:00,Житомирська,0,0.000000
384510,2023-12-23 15:00:00,8,47.8289,35.1626,2.7,1.0,88.4,3.900,100.0,50.00,...,15,Zaporozhye,Europe/Zaporozhye,2023-12-23,07:26:31,15:50:04,15:00:00,Запорізька,0,0.000000
181729,2023-01-05 13:00:00,3,50.7469,25.3263,6.5,4.0,84.4,7.124,100.0,8.33,...,13,Lutsk,Europe/Kiev,2023-01-05,08:20:08,16:28:05,13:00:00,Волинська,0,0.000000
373087,2023-12-03 19:00:00,9,48.9226,24.7147,-1.3,-3.6,84.3,0.900,100.0,29.17,...,19,Ivano-Frankivsk,Europe/Kiev,2023-12-03,07:55:33,16:25:52,19:00:00,Івано-Франківська,0,0.000000
425390,2024-03-03 14:00:00,17,50.6193,26.2513,2.7,-1.3,76.7,0.000,0.0,0.00,...,14,Rivne,Europe/Kiev,2024-03-03,06:54:07,18:00:28,14:00:00,Рівненська,0,0.000000
363049,2023-11-16 09:00:00,3,50.7469,25.3263,2.6,-0.3,82.1,0.918,100.0,4.17,...,9,Lutsk,Europe/Kiev,2023-11-16,07:35:03,16:31:13,09:00:00,Волинська,0,0.000000


In [5]:
df_final = df_final.sort_values(['region_id', 'datetime_hour'])
df_final.head()

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,hour,city_name,city_timezone,day_datetime,day_sunrise,day_sunset,hour_datetime,region_key,alarm_active,alarm_minutes_in_hour
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,00:00:00,Вінницька,0,0.0
24,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,1,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,01:00:00,Вінницька,0,0.0
48,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,2,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,02:00:00,Вінницька,0,0.0
72,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,3,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,03:00:00,Вінницька,0,0.0
96,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,4,Vinnytsia,Europe/Kiev,2022-02-24,06:58:49,17:40:52,04:00:00,Вінницька,0,0.0


In [6]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 634680 entries, 0 to 634679
Data columns (total 56 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   datetime_hour                  634680 non-null  datetime64[us]
 1   region_id                      634680 non-null  int64         
 2   city_latitude                  634680 non-null  float64       
 3   city_longitude                 634680 non-null  float64       
 4   day_temp                       634680 non-null  float64       
 5   day_dew                        634680 non-null  float64       
 6   day_humidity                   634680 non-null  float64       
 7   day_precip                     634680 non-null  float64       
 8   day_precipprob                 634680 non-null  float64       
 9   day_precipcover                634680 non-null  float64       
 10  day_snow                       634680 non-null  float64       
 11  day_snowdepth   

In [7]:
df_final = df_final.sort_values(["region_id", "datetime_hour"])

df_final["alarm_lag_1"] = df_final.groupby("region_id")["alarm_active"].shift(1)
df_final["alarm_lag_3"] = df_final.groupby("region_id")["alarm_active"].shift(3)
df_final["alarm_lag_6"] = df_final.groupby("region_id")["alarm_active"].shift(6)
df_final["alarm_lag_12"] = df_final.groupby("region_id")["alarm_active"].shift(12)

lag_cols = ["alarm_lag_1","alarm_lag_3","alarm_lag_6","alarm_lag_12"]
df_final[lag_cols] = df_final[lag_cols].fillna(0)

df_final['alarms_in_last_24h'] = (df_final.groupby('region_id')['alarm_active'].transform(lambda x: x.shift(1).rolling(24, min_periods=1).sum()))

df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)
df_final['is_night'] = ((df_final['hour'] >= 23) | (df_final['hour'] <= 6)).astype(int)

hourly_total = df_final.groupby('datetime_hour')['alarm_active'].sum().shift(1)
df_final['total_active_alarms_lag1'] = df_final['datetime_hour'].map(hourly_total)

df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarm_active,alarm_minutes_in_hour,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1
291701,2023-07-15 12:00:00,7,48.62640,22.28510,22.1,15.5,69.4,0.0,0.0,0.00,...,0,0.000000,0.0,0.0,0.0,0.0,2.0,1,0,1.0
365989,2023-11-21 11:00:00,16,49.58790,34.55170,-4.1,-6.7,82.5,0.0,0.0,0.00,...,0,0.000000,0.0,0.0,0.0,1.0,3.0,0,0,1.0
79310,2022-07-11 17:00:00,17,50.61927,26.25131,15.3,9.8,71.3,0.0,0.0,0.00,...,0,0.000000,0.0,0.0,0.0,1.0,6.0,0,0,0.0
625844,2025-02-14 15:00:00,23,49.44070,32.06370,0.0,-3.0,81.0,1.5,100.0,25.00,...,0,0.000000,0.0,0.0,0.0,0.0,6.0,0,0,3.0
280824,2023-06-26 15:00:00,2,49.23360,28.44860,20.0,13.3,68.6,0.5,100.0,4.17,...,0,0.000000,0.0,0.0,0.0,0.0,0.0,0,0,1.0
103464,2022-08-22 16:00:00,2,49.23360,28.44860,23.1,16.2,67.9,0.0,0.0,0.00,...,1,3.500000,1.0,0.0,1.0,0.0,2.0,0,0,24.0
109842,2022-09-02 17:00:00,21,46.64010,32.61420,20.8,6.6,41.3,0.0,0.0,0.00,...,0,0.000000,1.0,1.0,1.0,0.0,8.0,0,0,2.0
195147,2023-01-28 20:00:00,5,48.00200,37.81450,-3.3,-5.3,86.4,0.9,100.0,20.83,...,0,0.000000,0.0,0.0,0.0,1.0,8.0,1,0,0.0
135646,2022-10-17 12:00:00,25,51.49370,31.29440,8.4,2.9,70.9,0.0,0.0,0.00,...,1,18.166667,0.0,1.0,0.0,0.0,3.0,0,0,2.0
483603,2024-06-12 17:00:00,5,48.00200,37.81450,25.0,16.6,61.0,0.4,100.0,12.50,...,1,28.200000,1.0,1.0,1.0,1.0,22.0,0,0,11.0


In [8]:
neighbouring_regions = {
    1: [21],
    2: [6, 10, 11, 15, 22, 23, 24],
    3: [13, 17],
    4: [5, 8, 11, 14, 16, 20, 21],
    5: [4, 8, 12, 20],
    6: [2, 10, 17, 22],
    7: [9, 13],
    8: [4, 5, 21],
    9: [7, 13, 19, 24],
    10: [2, 6, 16, 23, 25],
    11: [2, 4, 14, 15, 16, 23],
    12: [5, 20],
    13: [3, 7, 9, 17, 19],
    14: [4, 11, 15, 21],
    15: [2, 11, 14],
    16: [4, 10, 11, 18, 20, 23, 25],
    17: [3, 6, 13, 19, 22],
    18: [16, 20, 25],
    19: [9, 13, 17, 22, 24],
    20: [4, 5, 12, 16, 18],
    21: [1, 4, 8, 14],
    22: [2, 6, 17, 19, 24],
    23: [2, 10, 11, 16],
    24: [2, 9, 19, 22],
    25: [10, 16, 18], 
    26: [10]
}

alarms_matrix = df_final.pivot_table(
    index='datetime_hour',
    columns='region_id',
    values='alarm_active',
    fill_value=0
)

neighbour_alarm_matrix = pd.DataFrame(index=alarms_matrix.index)

for region, neighbours in neighbouring_regions.items():
    valid_neighbours = [n for n in neighbours if n in alarms_matrix.columns]
    
    if valid_neighbours:
        neighbour_alarm_matrix[region] = alarms_matrix[valid_neighbours].sum(axis=1)
    else:
        neighbour_alarm_matrix[region] = 0

neighbour_alarm_matrix = neighbour_alarm_matrix.shift(1)

neighbour_alarm_long = (neighbour_alarm_matrix.stack().reset_index())

neighbour_alarm_long.columns = ['datetime_hour','region_id','neighbour_alarms']

df_final = df_final.merge(neighbour_alarm_long,on=['datetime_hour', 'region_id'], how='left')

In [9]:
def calculate_hours_since_last(series):
    series = series.shift(1).fillna(0)
    groups = series.cumsum()
    return series.groupby(groups).cumcount()

df_final['hours_since_last_alarm'] = (df_final.groupby('region_id')['alarm_active'].transform(calculate_hours_since_last))

In [10]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,alarm_lag_1,alarm_lag_3,alarm_lag_6,alarm_lag_12,alarms_in_last_24h,is_weekend,is_night,total_active_alarms_lag1,neighbour_alarms,hours_since_last_alarm
525561,2024-10-13 21:00:00,22,49.4168,26.9743,7.8,4.5,81.5,0.200,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,1,0,1.0,0.0,45
566078,2023-05-17 07:00:00,24,48.2924,25.9355,17.7,8.8,57.7,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,2.0,0,0,2.0,0.0,6
35983,2023-03-28 12:00:00,3,50.7469,25.3263,0.6,-3.9,73.9,13.447,100.0,8.33,...,0.0,0.0,0.0,0.0,1.0,0,0,2.0,0.0,12
400142,2022-07-18 12:00:00,18,50.9080,34.7972,16.1,11.8,77.5,1.025,100.0,8.33,...,0.0,0.0,0.0,0.0,1.0,0,0,2.0,0.0,9
509088,2022-11-27 10:00:00,22,49.4168,26.9743,0.4,-0.3,95.2,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,0.0,1,0,0.0,0.0,90
188820,2022-07-28 10:00:00,9,48.9226,24.7147,21.1,13.5,65.7,0.000,0.0,0.00,...,0.0,0.0,1.0,0.0,4.0,0,0,3.0,0.0,3
216097,2022-09-01 02:00:00,10,50.4506,30.5243,17.1,7.2,54.9,0.000,0.0,0.00,...,0.0,0.0,0.0,0.0,2.0,0,1,0.0,0.0,13
50696,2024-11-30 14:00:00,3,50.7469,25.3263,3.5,2.4,93.0,0.100,100.0,4.17,...,0.0,0.0,0.0,0.0,0.0,1,0,0.0,0.0,41
606781,2024-12-31 10:00:00,25,51.4937,31.2944,1.4,0.5,93.9,0.200,100.0,8.33,...,1.0,1.0,1.0,1.0,17.0,0,0,24.0,3.0,0
171949,2023-08-31 09:00:00,8,47.8289,35.1626,24.1,11.1,45.5,0.000,0.0,0.00,...,1.0,0.0,0.0,1.0,19.0,0,0,2.0,1.0,0


In [11]:
df_isw_matrix.sample(10)

,date,isw_topic_0,isw_topic_1,isw_topic_2,isw_topic_3,isw_topic_4,isw_topic_5,isw_topic_6,isw_topic_7,isw_topic_8,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
151,2022-07-24,0.700528,-0.228122,0.172324,-0.037730,-0.112951,0.163005,0.086332,-0.103761,-0.028997,...,-0.030757,-0.042649,0.001927,-0.011814,0.032448,0.009395,-0.004320,0.002549,-0.002520,-0.006079
1041,2025-01-09,0.810090,0.135044,-0.091436,0.057810,0.189314,0.143610,-0.089964,-0.137427,0.109052,...,0.006998,0.017136,-0.020290,-0.010803,-0.005741,0.004575,0.006211,0.004429,0.008906,-0.006182
749,2024-03-19,0.696374,-0.067375,-0.237334,0.232480,-0.003178,-0.099203,-0.044283,0.101224,0.094691,...,-0.018621,-0.005510,-0.021692,-0.002873,-0.007393,-0.029842,-0.010158,-0.018953,-0.016733,0.004956
898,2024-08-15,0.802696,0.025660,-0.040091,-0.032439,-0.170652,0.066417,-0.012253,0.082438,0.244065,...,0.018203,0.012575,-0.010322,0.012810,-0.015902,0.008564,-0.004819,0.014457,-0.006056,0.000179
1019,2024-12-16,0.746498,0.130148,-0.122388,0.016395,0.117843,-0.089719,0.322370,0.040489,-0.036480,...,0.021158,0.024860,-0.018181,0.008528,0.007206,-0.012451,0.002721,-0.001033,0.020516,0.007016
1216,2025-07-03,0.768030,0.312097,0.054703,0.036126,-0.181864,0.150062,0.071385,-0.046976,-0.093167,...,0.018764,0.003350,0.002493,-0.004315,0.005479,0.017649,-0.004960,0.001899,0.023237,0.006048
944,2024-09-30,0.788891,0.080691,-0.104995,-0.004693,-0.112658,-0.019008,-0.033391,-0.014243,0.095722,...,-0.024723,-0.014935,-0.005414,0.015946,-0.040397,0.007886,0.015934,0.021695,0.026705,-0.021471
571,2023-09-20,0.818030,-0.106034,-0.059142,-0.157102,-0.042887,0.017813,-0.049596,0.082784,0.005840,...,0.005994,-0.006298,0.012006,-0.014653,0.016700,-0.014819,0.000066,-0.024144,-0.030876,-0.001459
1442,2026-02-17,0.758643,0.247843,0.090029,-0.036350,0.097537,0.024765,-0.024823,0.126780,-0.079739,...,0.024101,0.008291,-0.042464,-0.001769,0.000112,-0.015796,0.032233,0.022377,-0.000321,0.044619
405,2023-04-07,0.774030,-0.227300,-0.065364,0.104712,0.003754,-0.034038,-0.030294,-0.009529,-0.119248,...,-0.014622,0.002759,-0.007078,0.005089,-0.016474,-0.016127,-0.000231,0.014363,-0.019356,0.034331


In [12]:
df_isw_matrix["date"] = pd.to_datetime(df_isw_matrix["date"]) + pd.Timedelta(days=1)
df_isw_matrix = df_isw_matrix.rename(columns={'date': 'day_datetime'})

In [13]:
df_final['day_datetime'] = pd.to_datetime(df_final['day_datetime']).dt.date
df_isw_matrix['day_datetime'] = pd.to_datetime(df_isw_matrix['day_datetime']).dt.date
df_isw_matrix.fillna(0, inplace=True)
df_isw_matrix.isna().sum()

day_datetime     0
isw_topic_0      0
isw_topic_1      0
isw_topic_2      0
isw_topic_3      0
                ..
isw_topic_145    0
isw_topic_146    0
isw_topic_147    0
isw_topic_148    0
isw_topic_149    0
Length: 151, dtype: int64

In [14]:
df_final = df_final.merge(df_isw_matrix, on="day_datetime", how="left")

In [15]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
440906,2024-02-18 04:00:00,19,49.5527,25.5889,2.6,-0.2,81.7,1.000,100.0,4.17,...,0.025668,0.012216,-0.010986,-0.031190,-0.002783,-0.015939,-0.035167,-0.008190,-0.017506,0.013171
510357,2022-12-30 07:00:00,22,49.4168,26.9743,2.6,-0.2,82.5,0.100,100.0,4.17,...,-0.004974,0.014419,-0.011882,0.015711,0.011208,0.011676,-0.005292,-0.002194,-0.027630,-0.018834
261350,2024-10-13 20:00:00,11,48.5085,32.2656,11.4,10.6,94.7,12.000,100.0,4.17,...,0.002451,-0.005775,-0.018439,0.005624,-0.003340,0.012910,0.021264,-0.020999,-0.006800,0.006210
554014,2024-12-15 13:00:00,23,49.4407,32.0637,0.0,-2.2,85.0,0.900,100.0,29.17,...,0.022313,0.022430,0.011422,-0.018059,0.005999,-0.002632,-0.021027,-0.006368,-0.012289,0.003041
407255,2023-04-24 22:00:00,18,50.9080,34.7972,11.5,5.1,67.3,0.982,100.0,4.17,...,-0.031145,-0.002229,0.008705,-0.015009,0.022004,-0.016291,0.029350,-0.019148,-0.010317,-0.002960
247493,2023-03-16 09:00:00,11,48.5085,32.2656,6.9,4.5,85.8,0.100,100.0,4.17,...,-0.010718,-0.005129,0.006521,-0.034959,0.004818,-0.022716,0.013500,-0.008038,0.011885,-0.015938
322524,2022-09-15 01:00:00,15,46.4725,30.7371,22.3,15.3,65.5,0.000,0.0,0.00,...,-0.012935,0.016432,-0.003560,0.008744,-0.056626,-0.002885,-0.023140,-0.025798,0.016618,-0.012940
285567,2024-07-12 00:00:00,13,49.8444,24.0254,25.4,18.6,68.6,0.800,100.0,4.17,...,0.037628,0.023348,-0.021502,0.005568,0.002932,-0.012651,-0.013204,-0.008840,0.012085,0.026527
113620,2023-01-11 17:00:00,6,50.2536,28.6654,-5.4,-7.5,85.3,5.000,100.0,4.17,...,0.002371,0.009834,-0.017741,-0.044539,-0.006734,0.004271,0.016606,0.005445,0.012817,0.019778
111014,2022-09-25 03:00:00,6,50.2536,28.6654,9.9,5.9,78.0,0.000,0.0,0.00,...,0.016653,-0.004069,0.015297,0.043231,-0.004206,0.033909,0.016727,0.020376,0.030997,-0.014075


In [16]:
df_final.isna().sum()

datetime_hour        0
region_id            0
city_latitude        0
city_longitude       0
day_temp             0
                  ... 
isw_topic_145     6336
isw_topic_146     6336
isw_topic_147     6336
isw_topic_148     6336
isw_topic_149     6336
Length: 216, dtype: int64

In [17]:
df_final.fillna(0, inplace=True)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_140,isw_topic_141,isw_topic_142,isw_topic_143,isw_topic_144,isw_topic_145,isw_topic_146,isw_topic_147,isw_topic_148,isw_topic_149
0,2022-02-24 00:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2022-02-24 01:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2022-02-24 02:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2022-02-24 03:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2022-02-24 04:00:00,2,49.2336,28.4486,2.8,-0.3,80.5,0.3,100.0,4.17,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635251,2025-03-01 19:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635252,2025-03-01 20:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635253,2025-03-01 21:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664
635254,2025-03-01 22:00:00,26,50.4506,30.5243,0.0,-2.6,82.9,0.7,100.0,8.33,...,-0.004258,0.016078,0.006146,0.014859,0.000348,-0.002017,0.002455,-0.008404,-0.019102,0.004664


In [18]:
df_final.isna().sum()

datetime_hour     0
region_id         0
city_latitude     0
city_longitude    0
day_temp          0
                 ..
isw_topic_145     0
isw_topic_146     0
isw_topic_147     0
isw_topic_148     0
isw_topic_149     0
Length: 216, dtype: int64

### Feature engineering for isw topics 

In [19]:
topic_cols = [c for c in df_final.columns if "isw_topic_" in c]
df_isw_abs = df_final[topic_cols].abs()

df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
df_final['isw_topic_max'] = df_isw_abs.max(axis=1)
df_final['isw_topic_mean'] = df_isw_abs.mean(axis=1)

df_final['isw_velocity_24h'] = df_final['isw_total_intensity'].diff(24)

df_final['isw_intensity_ema'] = (df_final.groupby('region_id')['isw_total_intensity'].transform(lambda x: x.shift(1).ewm(span=24).mean()))

df_final['isw_topic_entropy'] = -(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) * np.log(df_isw_abs.div(df_isw_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

isw_cols = ['isw_velocity_24h', 'isw_intensity_ema', 'isw_topic_entropy']
df_final[isw_cols] = df_final[isw_cols].fillna(0)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\1607078715.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_total_intensity'] = df_isw_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\1607078715.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['isw_topic_std'] = df_isw_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\1607078715.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [20]:
df_final.sample(10)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,isw_topic_147,isw_topic_148,isw_topic_149,isw_total_intensity,isw_topic_std,isw_topic_max,isw_topic_mean,isw_velocity_24h,isw_intensity_ema,isw_topic_entropy
261415,2024-10-16 13:00:00,11,48.5085,32.2656,9.3,5.8,79.9,0.1,100.0,4.17,...,-0.012813,0.016953,0.000472,4.444882,0.070499,0.810037,0.029633,-0.260585,4.536619,4.165249
79135,2025-02-18 16:00:00,4,48.4753,35.0160,-9.3,-13.2,74.7,0.0,0.0,0.00,...,-0.003565,-0.022910,-0.007679,5.064573,0.069890,0.787870,0.033764,0.603599,4.917037,4.279051
26684,2022-03-03 23:00:00,3,50.7469,25.3263,0.7,-1.7,84.3,0.6,100.0,8.33,...,-0.005982,-0.002720,0.005901,5.915269,0.065869,0.622981,0.039435,-0.196882,5.933259,4.343746
189073,2022-07-30 23:00:00,9,48.9226,24.7147,22.9,16.1,67.9,0.4,100.0,12.50,...,0.012948,-0.018922,-0.001243,5.026091,0.067279,0.730492,0.033507,0.132949,5.008139,4.263162
108942,2022-06-30 19:00:00,6,50.2536,28.6654,24.5,17.4,66.5,0.4,100.0,4.17,...,-0.001439,-0.004131,0.007696,5.890688,0.065277,0.692917,0.039271,0.809525,5.713281,4.392149
355464,2023-06-11 17:00:00,16,49.5879,34.5517,18.2,12.8,71.4,5.9,100.0,16.67,...,-0.014284,-0.004144,0.009157,4.793993,0.069429,0.789156,0.031960,-0.206250,4.843006,4.240515
2013,2022-05-17 22:00:00,2,49.2336,28.4486,14.4,1.7,44.8,0.0,0.0,0.00,...,0.014107,0.012890,-0.001514,4.469259,0.069515,0.736935,0.029795,-0.579951,4.559327,4.124621
593692,2023-06-12 00:00:00,25,51.4937,31.2944,13.9,7.6,67.5,0.9,100.0,12.50,...,-0.026735,-0.006051,-0.021118,5.174722,0.068512,0.774789,0.034498,0.380729,4.821335,4.317805
570596,2023-10-30 13:00:00,24,48.2924,25.9355,13.6,7.1,65.6,0.0,0.0,0.00,...,0.020167,0.000549,0.022831,4.899057,0.068965,0.797116,0.032660,-0.014734,4.894990,4.280514
408558,2023-06-18 05:00:00,18,50.9080,34.7972,21.7,11.7,56.2,0.0,0.0,0.00,...,-0.013788,-0.014273,0.014020,5.385316,0.067435,0.771852,0.035902,0.638745,4.922313,4.366942


### TELEGRAM MERGE

In [21]:
df_tg_matrix = pd.read_csv("../data/telegram_processed_svd.csv")

In [22]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,...,0.017141,-0.013944,-0.027516,0.034612,-0.032061,-0.051141,-0.004707,-0.003985,-0.052194,-0.026334
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077087,-0.078417,0.091534,...,-0.019097,0.025730,0.000530,0.020332,-0.028041,0.001581,0.006948,-0.064965,0.022597,-0.017810
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053647,-0.076998,0.085930,...,0.025378,-0.003550,-0.043415,-0.007200,-0.009172,0.006124,-0.004550,0.020576,0.010544,0.012825
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059696,...,0.008728,-0.007502,-0.021433,0.028223,-0.025323,-0.045761,-0.001697,0.002329,-0.042282,-0.017344
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089562,-0.088811,0.109836,...,0.017978,0.027453,-0.019744,0.008659,0.023139,-0.009485,-0.030263,0.013689,-0.022727,0.006843


In [23]:
df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")

C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\2851151304.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tg_matrix["datetime_hour"] = pd.to_datetime(df_tg_matrix["date"]).dt.floor("h")


In [24]:
topic_cols = [c for c in df_tg_matrix.columns if "tg_topic_" in c]
tg_hourly = (df_tg_matrix.groupby("datetime_hour")[topic_cols].mean().reset_index())

C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\873189955.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tg_hourly = (df_tg_matrix.groupby("datetime_hour")[topic_cols].mean().reset_index())


In [25]:
df_tg_matrix.head()

,date,channel,tg_topic_0,tg_topic_1,tg_topic_2,tg_topic_3,tg_topic_4,tg_topic_5,tg_topic_6,tg_topic_7,...,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,datetime_hour
0,2026-03-06 13:06:58,DeepStateUA,0.007121,0.000107,0.019843,0.000990,-0.011395,0.032670,-0.050983,0.061080,...,-0.013944,-0.027516,0.034612,-0.032061,-0.051141,-0.004707,-0.003985,-0.052194,-0.026334,2026-03-06 13:00:00
1,2026-03-06 12:57:29,DeepStateUA,0.022770,-0.002163,0.041469,0.001527,-0.033584,0.077087,-0.078417,0.091534,...,0.025730,0.000530,0.020332,-0.028041,0.001581,0.006948,-0.064965,0.022597,-0.017810,2026-03-06 12:00:00
2,2026-03-06 08:32:47,DeepStateUA,0.009647,-0.001083,0.029820,0.004092,-0.021362,0.053647,-0.076998,0.085930,...,-0.003550,-0.043415,-0.007200,-0.009172,0.006124,-0.004550,0.020576,0.010544,0.012825,2026-03-06 08:00:00
3,2026-03-05 16:46:58,DeepStateUA,0.006857,0.000279,0.019019,0.000875,-0.011544,0.032104,-0.049929,0.059696,...,-0.007502,-0.021433,0.028223,-0.025323,-0.045761,-0.001697,0.002329,-0.042282,-0.017344,2026-03-05 16:00:00
4,2026-03-05 15:46:39,DeepStateUA,0.048850,-0.006687,0.076040,-0.003924,-0.027197,0.089562,-0.088811,0.109836,...,0.027453,-0.019744,0.008659,0.023139,-0.009485,-0.030263,0.013689,-0.022727,0.006843,2026-03-05 15:00:00


In [26]:
tg_hourly["datetime_hour"] = tg_hourly["datetime_hour"] + pd.Timedelta(hours=1)

In [27]:
df_final = df_final.merge(tg_hourly, on="datetime_hour", how="left")

In [28]:
df_final.sample(20)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_240,tg_topic_241,tg_topic_242,tg_topic_243,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249
359179,2023-11-13 12:00:00,16,49.5879,34.5517,7.2,3.7,78.5,0.600,100.0,20.83,...,-0.004617,0.011168,0.001593,-0.004005,-0.007943,0.006727,-0.003674,-0.010468,-0.003654,0.001531
58056,2022-09-24 07:00:00,4,48.4753,35.0160,9.7,6.4,80.2,11.295,100.0,4.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89464,2023-04-18 03:00:00,5,48.0020,37.8145,10.2,1.9,56.4,1.400,100.0,20.83,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
235019,2024-10-19 14:00:00,10,50.4506,30.5243,4.9,1.3,78.0,0.000,0.0,0.00,...,0.005705,0.010523,-0.005750,-0.007900,-0.005052,0.004663,0.000288,0.007452,-0.004983,0.007761
247867,2023-04-01 00:00:00,11,48.5085,32.2656,8.7,6.3,85.9,2.000,100.0,4.17,...,-0.003995,0.002053,0.017670,-0.003237,-0.020976,-0.015314,0.005708,-0.000440,0.003678,0.003751
558046,2022-05-25 14:00:00,24,48.2924,25.9355,18.6,8.5,53.7,0.000,0.0,0.00,...,0.005168,0.005282,0.001564,0.021481,0.015545,0.004064,-0.011103,-0.003076,0.007177,-0.020312
197175,2023-07-03 14:00:00,9,48.9226,24.7147,21.0,14.9,70.9,3.000,100.0,4.17,...,0.020526,-0.020397,0.008495,-0.027207,-0.003653,0.032250,0.018072,0.027267,0.014329,-0.012638
104850,2025-01-18 06:00:00,5,48.0020,37.8145,0.4,-0.5,93.4,1.000,100.0,37.50,...,-0.005778,0.010839,0.011782,-0.000012,0.007621,-0.001295,-0.009310,0.008960,0.003963,-0.008264
101199,2024-08-19 03:00:00,5,48.0020,37.8145,23.6,6.2,37.2,0.000,0.0,0.00,...,0.009477,0.006074,0.001341,-0.001633,-0.003535,-0.010539,-0.003651,-0.012913,0.003408,-0.000270
60486,2023-01-03 13:00:00,4,48.4753,35.0160,6.2,4.9,91.8,0.000,0.0,0.00,...,-0.004901,0.004990,0.008875,-0.010320,-0.016325,0.003380,0.005471,0.010152,0.005595,0.007931


In [29]:
df_final.shape

(635256, 473)

In [30]:
tg_cols = [c for c in df_final.columns if 'tg_topic_' in c]
df_tg_abs = df_final[tg_cols].abs()
df_final[tg_cols] = df_final[tg_cols].ffill()

df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
df_final['tg_topic_max'] = df_tg_abs.max(axis=1)
df_final['tg_topic_entropy'] = -(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) * np.log(df_tg_abs.div(df_tg_abs.sum(axis=1), axis=0) + 1e-9)).sum(axis=1)

C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\3860349863.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_total_intensity'] = df_tg_abs.sum(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\3860349863.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['tg_topic_std'] = df_tg_abs.std(axis=1)
C:\Users\Uw11\AppData\Local\Temp\ipykernel_27696\3860349863.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

In [31]:
df_final.sample(15)

,datetime_hour,region_id,city_latitude,city_longitude,day_temp,day_dew,day_humidity,day_precip,day_precipprob,day_precipcover,...,tg_topic_244,tg_topic_245,tg_topic_246,tg_topic_247,tg_topic_248,tg_topic_249,tg_total_intensity,tg_topic_std,tg_topic_max,tg_topic_entropy
199420,2023-10-05 03:00:00,9,48.92260,24.71470,12.6,7.0,72.1,0.000,0.0,0.00,...,-0.000688,0.012820,0.013309,0.003012,0.004438,0.002227,4.590239,0.029313,0.269481,4.842698
492239,2023-12-13 07:00:00,21,46.64010,32.61420,5.1,4.5,96.1,0.100,100.0,4.17,...,0.023932,0.012357,-0.040690,-0.023804,-0.025204,-0.000364,6.737180,0.030778,0.189635,5.055141
354774,2023-05-13 23:00:00,16,49.58790,34.55170,15.4,4.0,50.0,0.000,0.0,0.00,...,0.000361,0.006105,-0.000822,-0.017845,0.007060,0.007295,4.051328,0.026194,0.218888,4.890113
429146,2022-10-16 03:00:00,19,49.55270,25.58890,8.9,3.0,67.5,0.000,0.0,0.00,...,-0.002356,-0.005142,-0.016717,0.020233,-0.038070,0.006851,0.000000,NaN,NaN,-0.000000
375971,2022-10-06 06:00:00,17,50.61927,26.25131,13.7,8.9,73.7,0.000,0.0,0.00,...,0.014167,0.001473,-0.030801,0.010136,-0.037758,-0.018579,0.000000,NaN,NaN,-0.000000
627645,2024-04-18 21:00:00,26,50.45060,30.52430,7.2,5.9,91.1,9.000,100.0,8.33,...,-0.009018,0.000459,0.013637,0.005771,-0.012879,0.005224,3.807484,0.015229,0.113537,5.118207
205465,2024-06-13 01:00:00,9,48.92260,24.71470,15.9,13.5,86.2,7.000,100.0,8.33,...,0.007632,0.030094,0.000179,0.002997,0.034663,0.001945,3.889683,0.032319,0.349052,4.653099
105611,2025-02-18 23:00:00,5,48.00200,37.81450,-10.4,-14.7,72.2,0.000,0.0,0.00,...,-0.012756,0.007741,-0.004758,0.010529,0.005628,-0.004526,2.832482,0.017736,0.169358,4.896658
423322,2025-02-22 10:00:00,18,50.90800,34.79720,-11.8,-15.2,76.6,0.000,0.0,0.00,...,-0.006716,0.000195,0.001055,0.007303,-0.002187,-0.003540,3.199356,0.020595,0.185845,4.873344
546689,2024-02-14 07:00:00,23,49.44070,32.06370,5.6,5.1,96.4,0.100,100.0,4.17,...,0.017187,-0.069873,0.022471,-0.052338,-0.021769,-0.013324,0.000000,NaN,NaN,-0.000000
